# Nim playground

Scratch notebook for trying things without the terminal. Pick the kernel **nim-arena-backend (.venv)**
(in VS Code: top-right *Select Kernel* → *Python Environments* → `backend/.venv`).
Top-level `await` works in cells.

In [3]:
%load_ext autoreload
%autoreload 2  # re-import edited app/ modules automatically

import pathlib
import sys

# Make `import app...` work no matter where the notebook is opened from.
root = pathlib.Path.cwd()
while not (root / "app").is_dir() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from app.agents import strategy
from app.agents.nim_optimal import NimOptimalAgent
from app.game.engine import GameConfig, PlayerSpec, apply_move, new_game

agent = NimOptimalAgent()


def make_state(piles, max_take=None, last_stone_loses=False, players=2):
    specs = [PlayerSpec(chr(97 + i), chr(65 + i)) for i in range(players)]  # a, b, c...
    return new_game(
        GameConfig(
            piles=list(piles), players=specs, max_take=max_take, last_stone_loses=last_stone_loses
        )
    )


async def ask(piles, **rules):
    """What would the agent play here? (first player to move)"""
    state = make_state(piles, **rules)
    move = await agent.choose_move(state, state.current_player.id)
    print(f"piles={list(piles)} {rules or ''}")
    print(f"  -> take {move.count} from pile {move.pile + 1}")
    print(f"  why: {move.reasoning}")
    return move

## One move

In [4]:
await ask([3, 4, 5])

piles=[3, 4, 5] 
  -> take 2 from pile 1
  why: Nim-sum of [3, 4, 5] is 2. I'm winning: take 2 from pile 1 (3→1).


AgentMove(pile=0, count=2, reasoning="Nim-sum of [3, 4, 5] is 2. I'm winning: take 2 from pile 1 (3→1).")

Same board with the other rule options:

In [5]:
await ask([3, 4, 5], max_take=3)
await ask([3, 4, 5], last_stone_loses=True)
await ask([1, 1, 3], last_stone_loses=True)
await ask([3, 4, 5], players=3)

piles=[3, 4, 5] {'max_take': 3}
  -> take 2 from pile 1
  why: With max 3 per move each pile counts as size mod 4: [3, 0, 1]; their XOR is 2. I'm winning: take 2 from pile 1 (3→1).
piles=[3, 4, 5] {'last_stone_loses': True}
  -> take 2 from pile 1
  why: Last stone loses. Nim-sum of [3, 4, 5] is 2. I'm winning: take 2 from pile 1 (3→1).
piles=[1, 1, 3] {'last_stone_loses': True}
  -> take 2 from pile 3
  why: Last stone loses. Nim-sum of [1, 1, 3] is 3. I'm winning: take 2 from pile 3 (3→1). That leaves 3 single stone(s), an odd number, so the opponent is forced to take the last one.
piles=[3, 4, 5] {'players': 3}
  -> take 2 from pile 1
  why: Nim-sum of [3, 4, 5] is 2. I'm winning: take 2 from pile 1 (3→1). (With 3 players this strategy is not optimal.)


AgentMove(pile=0, count=2, reasoning="Nim-sum of [3, 4, 5] is 2. I'm winning: take 2 from pile 1 (3→1). (With 3 players this strategy is not optimal.)")

## A whole game: the agent plays every seat

In [ ]:
async def play(piles, **rules):
    state = make_state(piles, **rules)
    print("start:", list(state.piles), rules or "")
    while not state.is_over:
        who = state.current_player.id
        move = await agent.choose_move(state, who)
        state = apply_move(state, who, move.pile, move.count)
        print(f"{who}: take {move.count} from pile {move.pile + 1} -> {list(state.piles)}")
    print("winner:", state.winner_id)
    return state


_ = await play([3, 4, 5], max_take=2, last_stone_loses=True)

start: [3, 4, 5] 
a: take 2 from pile 1 -> [1, 4, 5]
b: take 1 from pile 3 -> [1, 4, 4]
a: take 1 from pile 1 -> [0, 4, 4]
b: take 1 from pile 2 -> [0, 3, 4]
a: take 1 from pile 3 -> [0, 3, 3]
b: take 1 from pile 2 -> [0, 2, 3]
a: take 1 from pile 3 -> [0, 2, 2]
b: take 1 from pile 2 -> [0, 1, 2]
a: take 1 from pile 3 -> [0, 1, 1]
b: take 1 from pile 2 -> [0, 0, 1]
a: take 1 from pile 3 -> [0, 0, 0]
winner: a


## Is this position winning for the player to move?

In [7]:
strategy.evaluate((3, 4, 5), None, False)

Evaluation(method='nim_sum', winning=True, nim_sum=2, values=(3, 4, 5))

In [8]:
strategy.evaluate((1, 2, 3), None, False)  # nim-sum 0 -> losing

Evaluation(method='nim_sum', winning=False, nim_sum=0, values=(1, 2, 3))

## Your experiments

In [9]:
rules

NameError: name 'rules' is not defined

## Arena
Play agents against each other with no delay. Seats alternate between games.

In [4]:
from app.arena import format_result, run_arena

result = await run_arena("nim_optimal", "random", games=200, piles=[3, 4, 5], seed=1)
print(format_result(result))

200 games, average 6.5 moves
                      wins  win rate  when first
A: nim_optimal         200    100.0%     100/100
B: random                0      0.0%       0/100


In [ ]:
# 5.2: first LLM call (needs ANTHROPIC_API_KEY in backend/.env; costs a little)
from dotenv import load_dotenv
load_dotenv("../.env")  # notebook runs from backend/notebooks
from app.agents.anthropic_llm import AnthropicProvider
from app.agents.llm import AgentDecision

llm = AnthropicProvider().make().with_structured_output(AgentDecision, include_raw=True)
out = await llm.ainvoke("Nim: piles [3, 4, 5]. Take any number of stones from ONE pile; taking the last stone wins. You move. Pile numbers start at 1.")
print(out["parsed"])
print(out["raw"].usage_metadata)

In [ ]:
# 5.6 early: same call against the Modal-hosted open model (needs MODAL_LLM_URL / MODAL_LLM_API_KEY in .env).
# The first call after idle is a cold start (minutes).
from dotenv import load_dotenv
load_dotenv("../.env")
from app.agents.llm import AgentDecision
from app.agents.openai_compat_llm import OpenAICompatProvider

llm = OpenAICompatProvider().make().with_structured_output(AgentDecision, include_raw=True)
out = await llm.ainvoke("Nim: piles [3, 4, 5]. Take any number of stones from ONE pile; taking the last stone wins. You move. Pile numbers start at 1.\nAnswer ONLY with a JSON object with keys reasoning, pile, count.")
print(out["parsed"], out["parsing_error"])
print(out["raw"].usage_metadata)